<a href="https://colab.research.google.com/github/AI-Engineer-Abhi/Junior-AI-Engineer---Training/blob/main/Task-Day16/LLM_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q21. Using Hugging Face's transformers library, load the gpt2 tokenizer and tokenize the sentence "Large Language Models are transforming the world." Print the resulting tokens and their token IDs.

In [ ]:
from transformers import GPT2Tokenizer

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Input sentence
text = "Large Language Models are transforming the world."

# Tokenize the sentence
tokens = tokenizer.tokenize(text)

# Convert tokens to token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# Display the results
print("Original Text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Original Text:
Large Language Models are transforming the world.

Tokens:
['Large', 'ĠLanguage', 'ĠModels', 'Ġare', 'Ġtransforming', 'Ġthe', 'Ġworld', '.']

Token IDs:
[21968, 15417, 32329, 389, 25449, 262, 995, 13]


*Explanation of the Output*

---



The tokenizer does not simply split the sentence by spaces. Instead, it converts the text into tokens that exist in GPT-2's vocabulary.

Each token is assigned a unique integer called a Token ID, which is what the language model actually processes. These token IDs are later converted into embeddings before being fed into the Transformer model.

Q22. Write a Python script using AutoTokenizer to compare how the words "unbelievable" and "cat" are tokenized. Explain why one is split into multiple tokens and the other is not.

In [ ]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")             # AutoTokenizer automatically downloads the tokenizer for GPT-2.

# Words to compare
words = ["unbelievable", "cat"]

for word in words:
    print(f"\nWord: {word}")

    tokens = tokenizer.tokenize(word)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    print("Tokens :", tokens)
    print("Token IDs :", token_ids)


Word: unbelievable
Tokens : ['un', 'bel', 'iev', 'able']
Token IDs : [403, 6667, 11203, 540]

Word: cat
Tokens : ['cat']
Token IDs : [9246]


*Explanation of the Output*

---



The word "cat" is a common word that already exists in GPT-2's vocabulary, so it is represented by a single token.

The word "unbelievable" is less common as a complete token. Instead of creating a separate vocabulary entry for every possible word, GPT-2 uses Byte Pair Encoding (BPE) to split it into smaller subword units.

This approach helps the tokenizer efficiently represent rare or unseen words while keeping the vocabulary size manageable.

Q23. Using the transformers pipeline API, load gpt2 and generate text from the prompt "Artificial Intelligence is" with max_new_tokens=50. Run it twice and compare the outputs. Are they identical? Explain why or why not.

In [ ]:
from transformers import pipeline

# Load GPT-2 text generation pipeline
generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "Artificial Intelligence is"

# First generation
output1 = generator(
    prompt,
    max_new_tokens=50,
    do_sample=True
)

# Second generation
output2 = generator(
    prompt,
    max_new_tokens=50,
    do_sample=True
)

print("First Output:\n")
print(output1[0]["generated_text"])

print("\n")

print("Second Output:\n")
print(output2[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


First Output:

Artificial Intelligence is an international research and development initiative based in Beijing and led by IBM Research University. It is currently conducting research on artificial intelligence and the use of machine learning, machine learning algorithms, artificial intelligence and machine learning in the field of artificial intelligence in the fields of


Second Output:

Artificial Intelligence is a field of endeavor that includes artificial intelligence and machine learning. It is a field that encompasses the creation and development of artificial intelligence in the 21st century, artificial intelligence as a potential driver of the economy, and artificial intelligence as a potential tool to help


*Explanation of the Output*

---



The two generated outputs are usually different.

This happens because do_sample=True introduces randomness while selecting the next token. Instead of always choosing the highest-probability word, the model samples from multiple likely candidates.

Q24. Write code that pads a batch of three sentences of different lengths to the same length and prints the corresponding attention mask for each sentence.

In [ ]:
from transformers import AutoTokenizer

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Sentences with different lengths
sentences = [
    "I love AI.",
    "Machine learning is amazing.",
    "Transformers have changed natural language processing."
]

# Tokenize with padding                                                       # padding=True makes every sentence the same length.
                                                                              # Shorter sentences are padded with the PAD token (0).
                                                                              # attention_mask tells the Transformer which tokens are actual words and which are padding.
encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs:\n")
print(encoded["input_ids"])

print("\nAttention Mask:\n")
print(encoded["attention_mask"])

Input IDs:

tensor([[  101,  1045,  2293,  9932,  1012,   102,     0,     0,     0],
        [  101,  3698,  4083,  2003,  6429,  1012,   102,     0,     0],
        [  101, 19081,  2031,  2904,  3019,  2653,  6364,  1012,   102]])

Attention Mask:

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


*Explanation of the Output*

---


1 represents a real token.
0 represents a padding token.
During self-attention, the model ignores positions where the attention mask is 0.

Q25. Using a pretrained model (bert-base-uncased), extract the embedding vector for the word "bank" in two different sentences (financial vs. river context) and compare them numerically. What does this demonstrate about contextual embeddings?

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from torch.nn.functional import cosine_similarity

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

sentence1 = "I deposited money in the bank."
sentence2 = "The fisherman sat near the bank of the river."

# Function to get contextual embedding
def get_bank_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    bank_index = tokens.index("bank")

    embedding = outputs.last_hidden_state[0, bank_index]

    return embedding

emb1 = get_bank_embedding(sentence1)
emb2 = get_bank_embedding(sentence2)

similarity = cosine_similarity(
    emb1.unsqueeze(0),
    emb2.unsqueeze(0)
)

print("Cosine Similarity:")
print(similarity.item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine Similarity:
0.5025647282600403


*Explanation of the Output*

---



Although the word is the same, its meaning is different in both sentences.

The cosine similarity is less than 1, showing that BERT generates different embeddings depending on context.

This demonstrates contextual embeddings, one of the biggest improvements over Word2Vec and GloVe.

Q26. Using sentence-transformers (all-MiniLM-L6-v2), encode the sentences "Artificial Intelligence", "Machine Learning", and "I love pizza", then compute and print the cosine similarity matrix.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Artificial Intelligence",
    "Machine Learning",
    "I love pizza"
]

embeddings = model.encode(sentences, convert_to_tensor=True)

similarity = cos_sim(embeddings, embeddings)

print("Cosine Similarity Matrix:\n")
print(similarity)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cosine Similarity Matrix:

tensor([[1.0000, 0.7035, 0.1269],
        [0.7035, 1.0000, 0.0888],
        [0.1269, 0.0888, 1.0000]])


*Explanation of the Output*

---


Every sentence has similarity 1 with itself.
Artificial Intelligence and Machine Learning have high similarity because they are related topics.
I love pizza has much lower similarity because it belongs to a completely different context.

Q27. Implement scaled dot-product self-attention from scratch in PyTorch (without using nn.MultiheadAttention) for a small input matrix, and print the resulting attention weights and context vectors.

In [ ]:
import torch
import math

# Input embeddings (4 tokens, embedding size = 8)
X = torch.rand(4, 8)

# Create weight matrices
Wq = torch.rand(8, 8)
Wk = torch.rand(8, 8)
Wv = torch.rand(8, 8)

# Generate Query, Key and Value
Q = X @ Wq
K = X @ Wk
V = X @ Wv

# Step 1: Calculate attention scores
scores = Q @ K.T

# Step 2: Scale the scores
scores = scores / math.sqrt(K.shape[1])

# Step 3: Apply Softmax
attention_weights = torch.softmax(scores, dim=1)

# Step 4: Calculate Context Vectors
context_vectors = attention_weights @ V

print("Attention Scores:\n")
print(scores)

print("\nAttention Weights:\n")
print(attention_weights)

print("\nContext Vectors:\n")
print(context_vectors)

Attention Scores:

tensor([[ 8.7198,  6.8151,  8.6446, 11.4685],
        [ 6.7747,  5.2606,  6.6753,  8.8914],
        [ 9.9406,  7.7498,  9.8483, 13.0683],
        [11.0749,  8.6534, 11.0720, 14.5912]])

Attention Weights:

tensor([[0.0565, 0.0084, 0.0524, 0.8827],
        [0.0959, 0.0211, 0.0868, 0.7962],
        [0.0403, 0.0045, 0.0367, 0.9185],
        [0.0280, 0.0025, 0.0279, 0.9416]])

Context Vectors:

tensor([[3.0137, 2.2811, 2.0921, 1.9329, 2.3087, 2.1181, 2.4260, 1.4372],
        [2.9485, 2.2357, 2.0353, 1.8908, 2.2866, 2.0890, 2.3659, 1.4324],
        [3.0395, 2.2992, 2.1154, 1.9509, 2.3162, 2.1292, 2.4503, 1.4388],
        [3.0554, 2.3110, 2.1298, 1.9631, 2.3211, 2.1371, 2.4655, 1.4405]])


*Explanation of the Output*

---



The output contains three parts:

Input Matrix: The original token embeddings.
Attention Weights: Shows how much attention each token gives to every other token.
Context Vectors: The final representation produced after combining information from all tokens according to the attention weights.

Q28. Extend your self-attention implementation from single-head to multi-head attention with 4 heads for an embedding dimension of 32. Print the shape of the output after concatenation and the final linear projection.

In [ ]:
import torch
import math

# Parameters
embed_dim = 32
num_heads = 4
head_dim = embed_dim // num_heads

# Input (Batch=2, Sequence Length=5)
X = torch.rand(2, 5, embed_dim)

# Weight matrices
Wq = torch.rand(embed_dim, embed_dim)
Wk = torch.rand(embed_dim, embed_dim)
Wv = torch.rand(embed_dim, embed_dim)

# Generate Q, K, V
Q = X @ Wq
K = X @ Wk
V = X @ Wv

# Split into multiple heads
Q = Q.view(2, 5, num_heads, head_dim).transpose(1, 2)
K = K.view(2, 5, num_heads, head_dim).transpose(1, 2)
V = V.view(2, 5, num_heads, head_dim).transpose(1, 2)

# Scaled Dot-Product Attention
scores = Q @ K.transpose(-2, -1)
scores = scores / math.sqrt(head_dim)

weights = torch.softmax(scores, dim=-1)

head_output = weights @ V

# Concatenate all heads
output = head_output.transpose(1, 2).reshape(2, 5, embed_dim)

print("Output Shape:")
print(output.shape)

Output Shape:
torch.Size([2, 5, 32])


Q29. Implement a causal (look-ahead) mask for a 5-token sequence in PyTorch or NumPy, apply it to a raw attention score matrix, and print the masked scores before and after softmax.

In [ ]:
import torch

# Random attention scores
scores = torch.rand(5, 5)

print("Original Scores:\n")
print(scores)

# Create Upper Triangular Mask
mask = torch.triu(torch.ones(5, 5), diagonal=1)

print("\nCausal Mask:\n")
print(mask)

# Apply Mask
masked_scores = scores.masked_fill(mask == 1, float("-inf"))

print("\nScores After Applying Mask:\n")
print(masked_scores)

# Softmax
attention = torch.softmax(masked_scores, dim=1)

print("\nAttention Probabilities:\n")
print(attention)

Original Scores:

tensor([[0.8798, 0.2272, 0.5824, 0.2673, 0.7925],
        [0.6129, 0.1513, 0.7651, 0.2548, 0.8491],
        [0.2390, 0.9645, 0.6109, 0.7833, 0.7760],
        [0.4771, 0.0789, 0.7863, 0.9743, 0.8380],
        [0.4814, 0.9505, 0.7556, 0.0139, 0.9530]])

Causal Mask:

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

Scores After Applying Mask:

tensor([[0.8798,   -inf,   -inf,   -inf,   -inf],
        [0.6129, 0.1513,   -inf,   -inf,   -inf],
        [0.2390, 0.9645, 0.6109,   -inf,   -inf],
        [0.4771, 0.0789, 0.7863, 0.9743,   -inf],
        [0.4814, 0.9505, 0.7556, 0.0139, 0.9530]])

Attention Probabilities:

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6134, 0.3866, 0.0000, 0.0000, 0.0000],
        [0.2214, 0.4574, 0.3212, 0.0000, 0.0000],
        [0.2138, 0.1435, 0.2912, 0.3515, 0.0000],
        [0.1628, 0.2602, 0.2141, 0.1020, 0.2609]])


*Explanation of the Output*

---



The program first prints the original attention scores.

After applying the mask, all future positions become -∞, which means they are ignored.

Once Softmax is applied, those masked positions receive an attention value of 0.

This ensures that a token can only attend to itself and previous tokens, which is exactly how GPT performs next-token prediction.

Q30. Build a minimal GPT-style decoder block (masked multi-head attention + residual connection + LayerNorm + feed-forward network + residual connection + LayerNorm) in PyTorch, and run a random input tensor through it to verify the output shape matches the input shape.

In [ ]:
import torch
import torch.nn as nn

class DecoderBlock(nn.Module):

    def __init__(self):
        super().__init__()

        # Multi-Head Attention
        self.attention = nn.MultiheadAttention(
            embed_dim=32,
            num_heads=4,
            batch_first=True
        )

        # Layer Normalization
        self.norm1 = nn.LayerNorm(32)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # Second Layer Normalization
        self.norm2 = nn.LayerNorm(32)

    def forward(self, x):

        seq_length = x.size(1)

        # Create causal mask
        mask = torch.triu(
            torch.ones(seq_length, seq_length),
            diagonal=1
        ).bool()

        # Masked Multi-Head Attention
        attention_output, _ = self.attention(
            x, x, x,
            attn_mask=mask
        )

        # First Residual Connection + LayerNorm
        x = self.norm1(x + attention_output)

        # Feed Forward Network
        ffn_output = self.ffn(x)

        # Second Residual Connection + LayerNorm
        output = self.norm2(x + ffn_output)

        return output


# -----------------------------------------
# Test the Decoder Block
# -----------------------------------------
decoder = DecoderBlock()

input_tensor = torch.rand(2, 6, 32)

output = decoder(input_tensor)

print("Input Shape :", input_tensor.shape)
print("Output Shape:", output.shape)

Input Shape : torch.Size([2, 6, 32])
Output Shape: torch.Size([2, 6, 32])


*Explanation of the Output*

---

Input Shape : torch.Size([2, 6, 32])
The model receives a batch containing 2 sequences, each with 6 tokens and an embedding size of 32.

Output Shape : torch.Size([2, 6, 32])
The output shape is identical to the input shape because a Transformer decoder processes each token while preserving the sequence length and embedding dimension. Only the internal representations of the tokens are updated, making them richer and more context-aware.